In [1]:
# 추론 및 영상 분석

import cv2, pickle
import mediapipe as mp
import numpy as np
from collections import deque, defaultdict
from tensorflow.keras.models import load_model
from ultralytics import YOLO

# 설정값
SEQUENCE_LEN   = 15
CONF_THRESHOLD = 0.70
SMOOTH_WINDOW  = 3
OUTPUT_W       = 1920
OUTPUT_H       = 1080

FONT_SCALE  = 0.65
THICK       = 1
BOX_THICK   = 2
LINE_THICK  = 2
CIRCLE_OUT  = 4
CIRCLE_IN   = 3
HUD_SCALE   = 1.2
HUD_THICK   = 2

# 모델 로드
action_model = load_model('./model/action_model_v3.h5')
with open('./model/label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)
yolo    = YOLO('yolo11n.pt')
mp_pose = mp.solutions.pose
POSE_CONNECTIONS = list(mp_pose.POSE_CONNECTIONS)

def get_resize_info(w, h, target_w, target_h):
    scale_r = min(target_w / w, target_h / h)
    new_w = int(w * scale_r)
    new_h = int(h * scale_r)
    x_off = (target_w - new_w) // 2
    y_off = (target_h - new_h) // 2
    return scale_r, x_off, y_off

def transform_coords(x, y, scale_r, x_off, y_off):
    return int(x * scale_r + x_off), int(y * scale_r + y_off)

def draw_pose_on_canvas(canvas, landmarks, scale_r, x_off, y_off, crop_info):
    cx1, cy1, cx2, cy2 = crop_info
    crop_w = cx2 - cx1
    crop_h = cy2 - cy1

    pts = {}
    for idx, lm in enumerate(landmarks.landmark):
        abs_x = lm.x * crop_w + cx1
        abs_y = lm.y * crop_h + cy1
        tx, ty = transform_coords(abs_x, abs_y, scale_r, x_off, y_off)
        pts[idx] = (tx, ty, lm.visibility)

    for conn in POSE_CONNECTIONS:
        s, e = conn
        if s in pts and e in pts:
            sx, sy, sv = pts[s]; ex, ey, ev = pts[e]
            if sv > 0.3 and ev > 0.3:
                cv2.line(canvas, (sx, sy), (ex, ey), (255, 0, 0), LINE_THICK)

    for idx, (tx, ty, vis) in pts.items():
        if vis > 0.3:
            cv2.circle(canvas, (tx, ty), CIRCLE_OUT, (255, 255, 255), -1)
            cv2.circle(canvas, (tx, ty), CIRCLE_IN,  (0, 255, 0),   -1)

def draw_label_box(frame, text, x1, y1, color):
    (tw, th), bl = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, THICK)
    cv2.rectangle(frame, (x1, y1-th-bl-4), (x1+tw+4, y1), color, -1)
    cv2.putText(frame, text, (x1+2, y1-bl-2),
                cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, (255, 255, 255), THICK)

def draw_hud(frame, label, conf, person_conf):
    lines = [
        (f'{label.capitalize()} {conf:.2f}', (0, 0, 255)),
        (f'Person {person_conf:.2f}',        (0, 0, 255)),
    ]
    gap = int(HUD_SCALE * 40)
    for i, (text, clr) in enumerate(lines):
        y = int(HUD_SCALE * 50) + i * gap
        cv2.putText(frame, text, (20, y), cv2.FONT_HERSHEY_DUPLEX, HUD_SCALE, (0,0,0), HUD_THICK+4)
        cv2.putText(frame, text, (20, y), cv2.FONT_HERSHEY_DUPLEX, HUD_SCALE, clr, HUD_THICK)

def resize_with_padding(frame, target_w, target_h):
    h, w = frame.shape[:2]
    scale_r = min(target_w / w, target_h / h)
    new_w, new_h = int(w * scale_r), int(h * scale_r)
    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)
    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    x_off, y_off = (target_w - new_w) // 2, (target_h - new_h) // 2
    canvas[y_off:y_off+new_h, x_off:x_off+new_w] = resized
    return canvas

def extract_keypoints_crop(frame, x1, y1, x2, y2, pose):
    H, W = frame.shape[:2]
    pad = 20
    cx1, cy1 = max(0, x1-pad), max(0, y1-pad)
    cx2, cy2 = min(W, x2+pad), min(H, y2+pad)
    crop = frame[cy1:cy2, cx1:cx2]
    if crop.size == 0: return None, None, None
    result = pose.process(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    if not result.pose_landmarks: return None, None, None

    kp = []
    for lm in result.pose_landmarks.landmark:
        abs_x = lm.x * (cx2 - cx1) + cx1
        abs_y = lm.y * (cy2 - cy1) + cy1
        kp.extend([abs_x / W, abs_y / H, lm.z, lm.visibility])
    return np.array(kp, dtype=np.float32), result.pose_landmarks, (cx1, cy1, cx2, cy2)

# 개선된 ActionSmoother
class ActionSmoother:
    def __init__(self, w=SMOOTH_WINDOW):
        self.buf = deque(maxlen=w)

    def update(self, label, conf):
        self.buf.append((label, conf))

    def get(self):
        if not self.buf:
            return 'unknown', 0.0

        sm = defaultdict(float)
        for lbl, cf in self.buf:
            sm[lbl] += cf * cf  # confidence 제곱으로 가중치 부여

        best     = max(sm, key=lambda k: sm[k])
        avg_conf = float(np.mean([cf for lbl, cf in self.buf if lbl == best]))
        return best, avg_conf

def process_video(input_path, output_path):
    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    W_orig = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H_orig = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (OUTPUT_W, OUTPUT_H))

    scale_r, x_off, y_off = get_resize_info(W_orig, H_orig, OUTPUT_W, OUTPUT_H)

    seq_buf = deque(maxlen=SEQUENCE_LEN)
    smoother = ActionSmoother()
    last_label, last_conf = 'unknown', 0.0

    with mp_pose.Pose(model_complexity=1,
                      min_detection_confidence=0.5,
                      min_tracking_confidence=0.4) as pose:
        while True:
            ret, frame = cap.read()
            if not ret: break

            results = yolo.track(frame, classes=0, conf=0.5, persist=True, verbose=False)
            canvas  = resize_with_padding(frame, OUTPUT_W, OUTPUT_H)

            main_box_orig = None
            max_area = 0
            for box in results[0].boxes:
                ox1, oy1, ox2, oy2 = map(int, box.xyxy[0])
                area = (ox2-ox1)*(oy2-oy1)
                if area > max_area:
                    max_area = area
                    main_box_orig = (ox1, oy1, ox2, oy2, float(box.conf[0]))

            for box in results[0].boxes:
                ox1, oy1, ox2, oy2 = map(int, box.xyxy[0])
                bconf = float(box.conf[0])
                tx1, ty1 = transform_coords(ox1, oy1, scale_r, x_off, y_off)
                tx2, ty2 = transform_coords(ox2, oy2, scale_r, x_off, y_off)

                is_main = main_box_orig and (ox1, oy1, ox2, oy2) == main_box_orig[:4]
                color   = (0, 0, 255) if is_main else (255, 100, 0)
                cv2.rectangle(canvas, (tx1, ty1), (tx2, ty2), color, BOX_THICK)

                if not is_main:
                    draw_label_box(canvas, f'person {bconf:.2f}', tx1, ty1, (255, 100, 0))

            if main_box_orig:
                ox1, oy1, ox2, oy2, pconf = main_box_orig
                tx1, ty1 = transform_coords(ox1, oy1, scale_r, x_off, y_off)

                kp_res = extract_keypoints_crop(frame, ox1, oy1, ox2, oy2, pose)
                kp, landmarks, crop_coords_orig = kp_res

                if kp is not None:
                    draw_pose_on_canvas(canvas, landmarks, scale_r, x_off, y_off, crop_coords_orig)

                    seq_buf.append(kp)
                    if len(seq_buf) == SEQUENCE_LEN:
                        seq_in = np.array(seq_buf)[np.newaxis, ...]
                        pred   = action_model.predict(seq_in, verbose=0)
                        conf   = float(np.max(pred))
                        label  = le.inverse_transform([np.argmax(pred)])[0]
                        if conf >= CONF_THRESHOLD:
                            smoother.update(label, conf)
                        last_label, last_conf = smoother.get()

                action_text = f'{last_label} {last_conf:.2f}'
                person_text = f'person {pconf:.2f}'

                tw, th = cv2.getTextSize(person_text, cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, THICK)[0]
                draw_label_box(canvas, person_text, tx1, ty1, (0, 0, 255))
                draw_label_box(canvas, action_text, tx1, ty1 - th - 10, (0, 0, 255))
                draw_hud(canvas, last_label, last_conf, pconf)

            out.write(canvas)

    cap.release(); out.release()
    print(f"✅ 저장 완료: {output_path}")

# 실행
process_video(
    'C:/Users/user/Desktop/project_H/test_video/4.mp4',
    'C:/Users/user/Desktop/project_H/test_video_result/output_4.mp4'
)



✅ 저장 완료: C:/Users/user/Desktop/project_H/test_video_result/output_4.mp4


In [2]:
# 다중영상 분석

import os
os.makedirs('./test_video_result', exist_ok=True)

video_dir = './test_video'
for filename in os.listdir(video_dir):
    if filename.endswith('mp4'):
        input_path = os.path.join(video_dir, filename)
        output_path = f'./test_video_result/output_{filename}'
        print(f"\n🎬 처리 중: {filename}")
        process_video(input_path, output_path)

print("\n✅ 전체 저장 완료!")


🎬 처리 중: 1.mp4
✅ 저장 완료: ./test_video_result/output_1.mp4

🎬 처리 중: 10.mp4
✅ 저장 완료: ./test_video_result/output_10.mp4

🎬 처리 중: 13.mp4
✅ 저장 완료: ./test_video_result/output_13.mp4

🎬 처리 중: 14.mp4
✅ 저장 완료: ./test_video_result/output_14.mp4

🎬 처리 중: 15.mp4
✅ 저장 완료: ./test_video_result/output_15.mp4

🎬 처리 중: 16.mp4
✅ 저장 완료: ./test_video_result/output_16.mp4

🎬 처리 중: 18.mp4
✅ 저장 완료: ./test_video_result/output_18.mp4

🎬 처리 중: 21.mp4
✅ 저장 완료: ./test_video_result/output_21.mp4

🎬 처리 중: 23.mp4
✅ 저장 완료: ./test_video_result/output_23.mp4

🎬 처리 중: 24.mp4
✅ 저장 완료: ./test_video_result/output_24.mp4

🎬 처리 중: 25.mp4
✅ 저장 완료: ./test_video_result/output_25.mp4

🎬 처리 중: 26.mp4
✅ 저장 완료: ./test_video_result/output_26.mp4

🎬 처리 중: 28.mp4
✅ 저장 완료: ./test_video_result/output_28.mp4

🎬 처리 중: 3.mp4
✅ 저장 완료: ./test_video_result/output_3.mp4

🎬 처리 중: 4.mp4
✅ 저장 완료: ./test_video_result/output_4.mp4

🎬 처리 중: 5.mp4
✅ 저장 완료: ./test_video_result/output_5.mp4

🎬 처리 중: 6.mp4
✅ 저장 완료: ./test_video_result/output_6.mp4

🎬 처리 중